### **Number Theory for Cryptography**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Number_Theory_Foundations.ipynb)

> Public-key cryptography rests on a few number-theory operations. Some are **fast** (greatest common divisor, modular inverse, modular exponentiation) and some are believed to be **hard** for large numbers (factoring, discrete logarithm). This gap between easy and hard directions is what makes RSA, Diffie-Hellman, and ElGamal possible. This notebook implements each operation and measures how the hard ones scale.

#### **01. Euclidean Algorithm and Extended Euclidean Algorithm**

* `gcd(a, b) = gcd(b, a mod b)`, until the remainder is 0. It needs only O(log n) steps.
* The **extended** version also finds integers `x, y` with `a·x + b·y = gcd(a, b)` (Bézout's identity).

In [1]:
# Iterative Euclidean and extended Euclidean algorithms
import math
import time

def gcd(a, b):
    while b:
        a, b = b, a % b
    return abs(a)

def extended_gcd(a, b):
    """Returns (g, x, y) with a*x + b*y = g = gcd(a, b)."""
    old_r, r = a, b
    old_x, x = 1, 0
    old_y, y = 0, 1
    while r:
        q = old_r // r
        old_r, r = r, old_r - q * r
        old_x, x = x, old_x - q * x
        old_y, y = y, old_y - q * y
    return old_r, old_x, old_y

for a, b in [(24, 9), (7, 9), (240, 46)]:
    g, x, y = extended_gcd(a, b)
    print(f"gcd({a}, {b}) = {g} = {a}·({x}) + {b}·({y})")
    assert g == math.gcd(a, b) and a * x + b * y == g

gcd(24, 9) = 3 = 24·(-1) + 9·(3)
gcd(7, 9) = 1 = 7·(4) + 9·(-3)
gcd(240, 46) = 2 = 240·(-9) + 46·(47)


#### **02. Modular Inverse**

The inverse of `a` modulo `m` is the `x` with `a·x ≡ 1 (mod m)`. It exists only when `gcd(a, m) = 1`, and the extended Euclidean algorithm finds it directly. Trying every candidate takes up to `m` steps: **exponential in the number of bits** of `m`.

In [2]:
# Modular inverse: extended Euclid vs. brute force
def mod_inverse(a, m):
    g, x, _ = extended_gcd(a, m)
    if g != 1:
        raise ValueError(f"no inverse: gcd({a}, {m}) = {g}")
    return x % m

def mod_inverse_brute_force(a, m):
    return next(x for x in range(1, m) if a * x % m == 1)

a, m = 123_456, 10_000_019                     # m is prime
start = time.perf_counter(); fast = mod_inverse(a, m); t_fast = time.perf_counter() - start
start = time.perf_counter(); slow = mod_inverse_brute_force(a, m); t_slow = time.perf_counter() - start
print(f"inverse of {a} mod {m} = {fast} | check: {a * fast % m}")
print(f"extended Euclid: {t_fast * 1e6:.0f} µs | brute force: {t_slow:.2f} s")
assert fast == slow == pow(a, -1, m)

try:
    mod_inverse(94343442323232, 3144444444666)   # both even: the brute force would loop for days
except ValueError as error:
    print("Answered instantly:", error)

inverse of 123456 mod 10000019 = 2289160 | check: 1
extended Euclid: 5 µs | brute force: 0.11 s
Answered instantly: no inverse: gcd(94343442323232, 3144444444666) = 6


#### **03. Modular Exponentiation and the Discrete Logarithm**

* **Easy direction:** `y = g^x mod p` with square-and-multiply takes O(log x) multiplications. Python's `pow(g, x, p)` does it; `pow(g, x) % p` would first build a number with millions of digits.
* **Hard direction:** given `y`, find `x`. Brute force takes up to `p` steps; **baby-step giant-step** takes about `√p` steps and memory. For 2048-bit groups, both are infeasible.

In [3]:
# Discrete logarithm: brute force vs. baby-step giant-step
def dlog_brute_force(y, g, p):
    value = 1
    for x in range(p):
        if value == y:
            return x
        value = value * g % p
    return None

def dlog_bsgs(y, g, p):
    m = math.isqrt(p - 1) + 1
    baby = {pow(g, j, p): j for j in range(m)}          # g^j for j < m
    giant = pow(g, -m, p)                               # g^(-m)
    value = y
    for i in range(m):
        if value in baby:
            return i * m + baby[value]
        value = value * giant % p
    return None

p, g, secret = 9_048_610_007, 5, 25_000_000
y = pow(g, secret, p)
for name, method in [("brute force", dlog_brute_force), ("baby-step giant-step", dlog_bsgs)]:
    start = time.perf_counter()
    x = method(y, g, p)
    print(f"{name:<21}: x = {x} in {time.perf_counter() - start:.3f} s")
    assert pow(g, x, p) == y

brute force          : x = 25000000 in 2.208 s
baby-step giant-step : x = 25000000 in 0.141 s


#### **04. Factoring by Trial Division**

Multiplying two primes is instant; recovering them from the product by trial division takes about `√n` divisions, so **each extra bit of the primes doubles the work**.

In [4]:
# Trial division time for products of two primes of growing size
def factor_trial_division(n):
    if n % 2 == 0:
        return 2, n // 2
    for d in range(3, math.isqrt(n) + 1, 2):
        if n % d == 0:
            return d, n // d
    return n, 1

def next_prime(n):
    n += 1
    while any(n % d == 0 for d in range(2, math.isqrt(n) + 1)):
        n += 1
    return n

print(f"{'prime bits':>10} | {'n':>22} | {'seconds':>8}")
for bits in (16, 20, 24, 26):
    p1, p2 = next_prime(2 ** bits - 2 ** (bits // 2)), next_prime(2 ** bits - 2 ** (bits // 3))
    start = time.perf_counter()
    found = factor_trial_division(p1 * p2)
    elapsed = time.perf_counter() - start
    assert sorted(found) == sorted((p1, p2))
    print(f"{bits:>10} | {p1 * p2:>22} | {elapsed:>8.3f}")
print("\nRSA uses primes of 1024 bits or more: trial division would need about 2^1023 steps.")

prime bits |                      n |  seconds
        16 |             4277538953 |    0.001
        20 |          1098383420003 |    0.019
        24 |        281402164634891 |    0.308
        26 |       4503034975279073 |    1.175

RSA uses primes of 1024 bits or more: trial division would need about 2^1023 steps.


#### **Key Takeaways**

* GCD, modular inverse, and modular exponentiation are fast (logarithmic in the size of the numbers).
* Factoring and the discrete logarithm grow exponentially with brute force; better algorithms exist, which is why key sizes are large.
* Always use `pow(g, x, p)` (or `pow(a, -1, m)` for inverses), never `pow(g, x) % p`.